In [1]:
import pythainlp as ptn

sentence = "วันนี้เราจะกินอะไรก่อนกาลดี"
words = ptn.word_tokenize(sentence)
print(words)

['วันนี้', 'เรา', 'จะ', 'กิน', 'อะไร', 'ก่อน', 'กาล', 'ดี']


In [2]:
# Turn words into embeddings
import pythainlp as ptn

sentence = "วันนี้เราจะกินอะไรก่อนกาลดี"
words = ptn.word_tokenize(sentence)


AttributeError: module 'pythainlp' has no attribute 'word_vector_service'

In [3]:
"""
Thai Text Preprocessing Pipeline for Chatterbox TTS Model Training
==================================================================

This pipeline converts Thai text into embeddings suitable for TTS model training,
specifically optimized for models like Chatterbox TTS.
"""

import re
import json
import numpy as np
from typing import List, Dict, Tuple
import torch
from dataclasses import dataclass

# Install required packages:
# pip install pythainlp epitran phonemizer transformers

# Thai-specific imports
from pythainlp.tokenize import word_tokenize
from pythainlp.util import normalize as thai_normalize
from pythainlp.transliterate import romanize, transliterate
from pythainlp.util import digit_to_text, num_to_thaiword
import pythainlp

# For phoneme conversion
from phonemizer import phonemize
from phonemizer.backend import EspeakBackend
from phonemizer.separator import Separator

@dataclass
class TTSDataSample:
    """Data structure for TTS training samples"""
    text: str
    normalized_text: str
    phonemes: str
    tokens: List[str]
    audio_path: str
    duration: float = None
    
class ThaiTTSPreprocessor:
    """
    Thai Text Preprocessing Pipeline for TTS Model Training
    
    This preprocessor handles:
    1. Text normalization (numbers, dates, symbols)
    2. Thai word segmentation
    3. Phoneme conversion
    4. Text embedding preparation
    """
    
    def __init__(self, 
                 use_phonemes: bool = True,
                 phoneme_language: str = 'th',
                 max_text_length: int = 512):
        self.use_phonemes = use_phonemes
        self.phoneme_language = phoneme_language
        self.max_text_length = max_text_length
        
        # Initialize phonemizer backend for Thai
        if use_phonemes:
            self.phonemizer_backend = EspeakBackend(
                language=phoneme_language,
                preserve_punctuation=True,
                with_stress=True
            )
            
        # Character/phoneme vocabulary (extend as needed)
        self.build_vocabulary()
        
    def build_vocabulary(self):
        """Build character and phoneme vocabulary for Thai"""
        # Thai characters
        thai_chars = 'กขฃคฅฆงจฉชซฌญฎฏฐฑฒณดตถทธนบปผฝพฟภมยรลวศษสหฬอฮ'
        thai_vowels = 'ะาำิีึืุูเแโใไๅ็่้๊๋์'
        thai_tones = '่้๊๋'
        thai_symbols = 'ๆฯ'
        
        # Common phonemes for Thai (IPA symbols)
        thai_phonemes = ['p', 'pʰ', 'b', 't', 'tʰ', 'd', 'k', 'kʰ', 'ʔ', 
                        'm', 'n', 'ŋ', 'r', 'l', 'w', 'j', 's', 'h',
                        'a', 'aː', 'i', 'iː', 'ɯ', 'ɯː', 'u', 'uː',
                        'e', 'eː', 'ɛ', 'ɛː', 'o', 'oː', 'ɔ', 'ɔː']
        
        # Build complete vocabulary
        all_chars = list(thai_chars + thai_vowels + thai_symbols)
        all_chars.extend(list('abcdefghijklmnopqrstuvwxyz'))
        all_chars.extend(list('0123456789'))
        all_chars.extend([' ', ',', '.', '!', '?', '-', ':', ';', '"', "'"])
        
        # Special tokens
        special_tokens = ['<pad>', '<sos>', '<eos>', '<unk>']
        
        self.vocab = special_tokens + all_chars + thai_phonemes
        self.char_to_id = {char: idx for idx, char in enumerate(self.vocab)}
        self.id_to_char = {idx: char for idx, char in enumerate(self.vocab)}
        
    def normalize_thai_text(self, text: str) -> str:
        """
        Normalize Thai text for TTS processing
        
        Steps:
        1. Convert numbers to Thai words
        2. Normalize Thai characters
        3. Remove excessive spaces
        4. Handle special symbols
        """
        # Normalize Thai characters (e.g., ํา -> ำ)
        text = thai_normalize(text)
        
        # Convert Arabic numerals to Thai words
        text = self._convert_numbers_to_words(text)
        
        # Remove multiple spaces
        text = re.sub(r'\s+', ' ', text)
        
        # Remove or convert special characters
        text = self._handle_special_chars(text)
        
        return text.strip()
    
    def _convert_numbers_to_words(self, text: str) -> str:
        """Convert numbers in text to Thai words"""
        # Find all numbers in the text
        numbers = re.findall(r'\d+', text)
        
        for num in numbers:
            try:
                # Convert to Thai word
                thai_word = num_to_thaiword(int(num))
                text = text.replace(num, thai_word)
            except:
                # For very large numbers or decimals, use digit_to_text
                thai_digits = digit_to_text(num)
                text = text.replace(num, thai_digits)
                
        return text
    
    def _handle_special_chars(self, text: str) -> str:
        """Handle special characters and symbols"""
        # Common replacements
        replacements = {
            '&': 'และ',
            '%': 'เปอร์เซ็นต์',
            '$': 'ดอลลาร์',
            '€': 'ยูโร',
            '£': 'ปอนด์',
            '@': 'แอท',
            '#': 'แฮช',
            '+': 'บวก',
            '-': 'ลบ',
            '×': 'คูณ',
            '÷': 'หาร',
            '=': 'เท่ากับ'
        }
        
        for old, new in replacements.items():
            text = text.replace(old, new)
            
        # Remove other non-Thai/English characters
        text = re.sub(r'[^\u0E00-\u0E7F\u0020-\u007E\s]', '', text)
        
        return text
    
    def segment_thai_words(self, text: str) -> List[str]:
        """
        Segment Thai text into words using PyThaiNLP
        
        Thai doesn't use spaces between words, so segmentation is crucial
        """
        # Use newmm engine for better accuracy
        words = word_tokenize(text, engine='newmm', keep_whitespace=True)
        
        # Filter out empty tokens
        words = [w for w in words if w.strip()]
        
        return words
    
    def text_to_phonemes(self, text: str) -> str:
        """
        Convert Thai text to phonemes using G2P
        
        Options:
        1. Using phonemizer with espeak-ng (requires Thai support)
        2. Using PyThaiNLP's transliterate function
        3. Using custom Thai G2P model
        """
        if not self.use_phonemes:
            return text
            
        try:
            # Method 1: Using phonemizer (requires espeak-ng with Thai)
            separator = Separator(phone=' ', word=' | ', syllable='')
            phonemes = phonemize(
                text,
                language=self.phoneme_language,
                backend='espeak',
                separator=separator,
                strip=True
            )
            
        except:
            # Method 2: Fallback to PyThaiNLP romanization/transliteration
            # This gives Thai romanization which can serve as pseudo-phonemes
            phonemes = transliterate(text, engine='thaig2p')
            
            # Alternative: Use IPA transliteration
            # phonemes = transliterate(text, engine='ipa')
            
        return phonemes
    
    def prepare_text_embeddings(self, text: str) -> Dict[str, np.ndarray]:
        """
        Prepare text embeddings for TTS model training
        
        Returns:
        - character_ids: Character-level token IDs
        - phoneme_ids: Phoneme-level token IDs (if enabled)
        - text_mask: Attention mask for padding
        """
        # Step 1: Normalize text
        normalized_text = self.normalize_thai_text(text)
        
        # Step 2: Get word segments
        words = self.segment_thai_words(normalized_text)
        
        # Step 3: Convert to phonemes
        phonemes = self.text_to_phonemes(normalized_text) if self.use_phonemes else ""
        
        # Step 4: Convert to token IDs
        # Character-level tokenization
        char_ids = self._text_to_ids(normalized_text)
        
        # Phoneme-level tokenization
        phoneme_ids = self._phonemes_to_ids(phonemes) if self.use_phonemes else []
        
        # Step 5: Pad sequences
        char_ids = self._pad_sequence(char_ids, self.max_text_length)
        phoneme_ids = self._pad_sequence(phoneme_ids, self.max_text_length)
        
        # Create attention mask
        text_mask = np.array([1 if idx != self.char_to_id['<pad>'] else 0 
                             for idx in char_ids])
        
        return {
            'text': text,
            'normalized_text': normalized_text,
            'words': words,
            'phonemes': phonemes,
            'character_ids': np.array(char_ids),
            'phoneme_ids': np.array(phoneme_ids),
            'text_mask': text_mask,
            'text_length': len(normalized_text)
        }
    
    def _text_to_ids(self, text: str) -> List[int]:
        """Convert text to character IDs"""
        ids = []
        for char in text:
            if char in self.char_to_id:
                ids.append(self.char_to_id[char])
            else:
                ids.append(self.char_to_id['<unk>'])
        return ids
    
    def _phonemes_to_ids(self, phonemes: str) -> List[int]:
        """Convert phonemes to IDs"""
        # Split phonemes by space
        phoneme_list = phonemes.split()
        ids = []
        
        for phoneme in phoneme_list:
            if phoneme in self.char_to_id:
                ids.append(self.char_to_id[phoneme])
            else:
                # Try to find closest match or use unknown
                ids.append(self.char_to_id['<unk>'])
                
        return ids
    
    def _pad_sequence(self, sequence: List[int], max_length: int) -> List[int]:
        """Pad or truncate sequence to fixed length"""
        if len(sequence) < max_length:
            # Pad with <pad> token
            sequence = sequence + [self.char_to_id['<pad>']] * (max_length - len(sequence))
        else:
            # Truncate
            sequence = sequence[:max_length]
            
        return sequence
    
    def prepare_dataset(self, data_files: List[Dict[str, str]]) -> List[Dict]:
        """
        Prepare complete dataset for TTS training
        
        Args:
            data_files: List of dicts with 'text' and 'audio_path' keys
            
        Returns:
            List of processed samples ready for training
        """
        processed_samples = []
        
        for sample in data_files:
            # Get embeddings
            embeddings = self.prepare_text_embeddings(sample['text'])
            
            # Create training sample
            processed_sample = {
                'id': sample.get('id', f'sample_{len(processed_samples)}'),
                'raw_text': sample['text'],
                'normalized_text': embeddings['normalized_text'],
                'phonemes': embeddings['phonemes'],
                'character_ids': embeddings['character_ids'].tolist(),
                'phoneme_ids': embeddings['phoneme_ids'].tolist(),
                'text_mask': embeddings['text_mask'].tolist(),
                'text_length': embeddings['text_length'],
                'audio_path': sample['audio_path'],
                'duration': sample.get('duration', None)
            }
            
            processed_samples.append(processed_sample)
            
        return processed_samples

# Example usage
def main():
    

if __name__ == "__main__":
    main()

Original text: สวัสดีครับ วันนี้อากาศดีมาก อุณหภูมิ 25 องศา
Normalized text: สวัสดีครับ วันนี้อากาศดีมาก อุณหภูมิ ยี่สิบห้า องศา
Words: ['สวัสดี', 'ครับ', 'วันนี้', 'อากาศ', 'ดีมาก', 'อุณหภูมิ', 'ยี่', 'สิบห้า', 'องศา']
Phonemes: s a5 w m s a2 d s k h aɜ r m b | w m n a2 n s | ʔ aɜ s k a5 s s a2 d s m aɜ s k | ʔ u2 n a5 h a2 p h uː2 m i | j s | s iɜ b a5 h | s | ʔ a2 n ɡ a5 s a s
Character IDs shape: (512,)
Text mask shape: (512,)

Processed 3 samples


In [5]:
# Initialize preprocessor
preprocessor = ThaiTTSPreprocessor(use_phonemes=True)

# Example Thai text
sample_text = "สวัสดีครับ วันนี้อากาศดีมาก อุณหภูมิ 25 องศา"

# Process single text
embeddings = preprocessor.prepare_text_embeddings(sample_text)

print("Original text:", sample_text)
print("Normalized text:", embeddings['normalized_text'])
print("Words:", embeddings['words'])
print("Phonemes:", embeddings['phonemes'])
print("Character IDs shape:", embeddings['character_ids'].shape)
print("Text mask shape:", embeddings['text_mask'].shape)

# Prepare dataset
dataset = [
    {'text': 'สวัสดีครับ', 'audio_path': 'audio1.wav'},
    {'text': 'ขอบคุณค่ะ', 'audio_path': 'audio2.wav'},
    {'text': 'ราคา 100 บาท', 'audio_path': 'audio3.wav'}
]

processed_dataset = preprocessor.prepare_dataset(dataset)

# Save to JSON for training
with open('thai_tts_dataset.json', 'w', encoding='utf-8') as f:
    json.dump(processed_dataset, f, ensure_ascii=False, indent=2)

print(f"\nProcessed {len(processed_dataset)} samples")

Original text: สวัสดีครับ วันนี้อากาศดีมาก อุณหภูมิ 25 องศา
Normalized text: สวัสดีครับ วันนี้อากาศดีมาก อุณหภูมิ ยี่สิบห้า องศา
Words: ['สวัสดี', 'ครับ', 'วันนี้', 'อากาศ', 'ดีมาก', 'อุณหภูมิ', 'ยี่', 'สิบห้า', 'องศา']
Phonemes: s a5 w m s a2 d s k h aɜ r m b | w m n a2 n s | ʔ aɜ s k a5 s s a2 d s m aɜ s k | ʔ u2 n a5 h a2 p h uː2 m i | j s | s iɜ b a5 h | s | ʔ a2 n ɡ a5 s a s
Character IDs shape: (512,)
Text mask shape: (512,)

Processed 3 samples


In [6]:
embeddings


{'text': 'สวัสดีครับ วันนี้อากาศดีมาก อุณหภูมิ 25 องศา',
 'normalized_text': 'สวัสดีครับ วันนี้อากาศดีมาก อุณหภูมิ ยี่สิบห้า องศา',
 'words': ['สวัสดี',
  'ครับ',
  'วันนี้',
  'อากาศ',
  'ดีมาก',
  'อุณหภูมิ',
  'ยี่',
  'สิบห้า',
  'องศา'],
 'phonemes': 's a5 w m s a2 d s k h aɜ r m b | w m n a2 n s | ʔ aɜ s k a5 s s a2 d s m aɜ s k | ʔ u2 n a5 h a2 p h uː2 m i | j s | s iɜ b a5 h | s | ʔ a2 n ɡ a5 s a s',
 'character_ids': array([ 43,  40,   3,  43,  23,  52,   7,  38,   3,  29, 107,  40,   3,
         28,  28,  52,  65,  46,  49,   4,  49,  41,  23,  52,  36,  49,
          4, 107,  46,  55,  22,  44,  35,  56,  36,  51, 107,  37,  52,
         64,  43,  51,  29,  44,  65,  49, 107,  46,  10,  41,  49,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   